# Evaluation der LLM (Ollama) PDF-Extraktion

Dieses Notebook dient zur Auswertung, wie gut die Extraktion von Abstracts und Keywords via Ollama funktioniert im Vergleich zur reinen Regex-Extraktion.

**Voraussetzungen**:
1. Wir benötigen einen Datensatz, der **Ground Truth** Abstracts (z.B. durch Semantic Scholar / OpenAlex) enthält.
2. Wir vergleichen diese Ground Truth mit dem Text, den Ollama/Regex aus der PDF geholt haben.

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from difflib import SequenceMatcher
import os
import sys
sys.path.append(os.path.abspath('../'))
from litresearch.extractors.pdf_extractor import PDFExtractor

## 1. Daten laden

Hier laden wir den Datensatz. Wir nehmen an, dass es eine Spalte `abstract` (Ground Truth von S2/OpenAlex) und eine Spalte `ee` oder `url` (PDF-Link) gibt.

In [6]:
print(df["extraction_source"].value_counts())
print(f"\nAbstract gefunden: {df['abstract_pdf'].notna().sum()} / {len(df)}")
print(f"Keywords gefunden: {df['keywords_pdf'].notna().sum()} / {len(df)}")

# Plausibilitätschecks: Längenverteilung
df["abstract_len"] = df["abstract_pdf"].str.len()
df["keywords_len"] = df["keywords_pdf"].str.len()
print(df[["abstract_len", "keywords_len"]].describe())

# Ausreißer: zu kurze oder zu lange Abstracts
suspicious = df[(df["abstract_len"] < 100) | (df["abstract_len"] > 3000)]
print(f"\nVerdächtige Abstracts: {len(suspicious)}")


NameError: name 'df' is not defined

In [ ]:
sample = (
    df.groupby("extraction_source", group_keys=False)
    .apply(lambda g: g.sample(min(10, len(g)), random_state=42))
    .reset_index(drop=True)
)

# Nur relevante Spalten für manuelle Prüfung
cols = ["title", "doi", "ee", "extraction_source", "abstract_pdf", "keywords_pdf"]
sample_export = sample[cols].copy()

# Spalten für manuelle Bewertung hinzufügen
sample_export["abstract_correct"] = ""   # 1 = korrekt, 0 = falsch, 0.5 = teilweise
sample_export["keywords_correct"] = ""
sample_export["notes"] = ""

sample_export.to_csv("manual_evaluation_sample.csv", index=False)
print(f"Sample exportiert: {len(sample_export)} Paper")
print(sample_export["extraction_source"].value_counts())

In [ ]:
gt = pd.read_csv("gt.csv", delimiter=";")
pred = sample_export.copy()

In [ ]:
import pandas as pd
import numpy as np
from difflib import SequenceMatcher
import re

# ── MERGE ───────────────────────────────────────────────────────────────────
merged = pd.merge(
    gt.rename(columns={"keywords": "keywords_gt"}).drop(columns=["extraction_source"]),
    pred[["ee", "extraction_source", "keywords_pdf"]],
    on="ee",
    how="inner"
)
print(f"Gematcht: {len(merged)} Paper\n")

# ── HILFSFUNKTIONEN ─────────────────────────────────────────────────────────
def normalize(text):
    if pd.isna(text) or str(text).strip() == "":
        return ""
    return re.sub(r"\s+", " ", str(text).lower().strip())

def similarity(a, b):
    a, b = normalize(a), normalize(b)
    if not a and not b: return 1.0
    if not a or not b:  return 0.0
    return SequenceMatcher(None, a, b).ratio()

def jaccard(a, b):
    sa, sb = set(normalize(a).split()), set(normalize(b).split())
    if not sa and not sb: return 1.0
    if not sa or not sb:  return 0.0
    return len(sa & sb) / len(sa | sb)

# ── METRIKEN ────────────────────────────────────────────────────────────────
merged["sim"]     = merged.apply(lambda r: similarity(r["keywords_gt"], r["keywords_pdf"]), axis=1)
merged["jaccard"] = merged.apply(lambda r: jaccard(r["keywords_gt"], r["keywords_pdf"]), axis=1)
merged["exact"]   = merged.apply(lambda r: int(normalize(r["keywords_gt"]) == normalize(r["keywords_pdf"])), axis=1)
merged["missing"] = merged["keywords_pdf"].apply(lambda x: pd.isna(x) or str(x).strip() == "")

# ── SUMMARY ─────────────────────────────────────────────────────────────────
print("GESAMT")
print(f"  ø Similarity:  {merged['sim'].mean():.3f}")
print(f"  ø Jaccard:     {merged['jaccard'].mean():.3f}")
print(f"  Exact Match:   {merged['exact'].mean()*100:.1f}%")
print(f"  Missing:       {merged['missing'].mean()*100:.1f}%")

print("\nNACH EXTRACTION SOURCE")
print(
    merged.groupby("extraction_source")
    .agg(
        n             = ("sim", "count"),
        sim_mean      = ("sim", "mean"),
        jaccard_mean  = ("jaccard", "mean"),
        exact_pct     = ("exact", lambda x: f"{x.mean()*100:.1f}%"),
        missing_pct   = ("missing", lambda x: f"{x.mean()*100:.1f}%"),
    )
    .round(3)
    .to_string()
)

print("\nPROBLEM-FÄLLE (sim < 0.60)")
print(
    merged[merged["sim"] < 0.60]
    [["extraction_source", "keywords_gt", "keywords_pdf", "sim"]]
    .to_string()
)


In [ ]:
missing = df[df["keywords_pdf"].isna() | (df["keywords_pdf"].str.strip() == "")]
print(f"Missing gesamt: {len(missing)} / {len(df)} ({len(missing)/len(df)*100:.1f}%)")
print(missing["extraction_source"].value_counts())


In [ ]:
# Heuristik: "Kein Keyword" ist suspekt wenn Abstract vorhanden
suspicious = df[
    (df["keywords_pdf"].isna() | (df["keywords_pdf"].str.strip() == "")) &
    (df["abstract_pdf"].notna() & (df["abstract_pdf"].str.strip() != ""))
]
print(f"Missing Keywords aber Abstract vorhanden: {len(suspicious)}")
# → Diese sind wahrscheinlich echte Pipeline-Fehler

no_abstract_either = df[
    (df["keywords_pdf"].isna() | (df["keywords_pdf"].str.strip() == "")) &
    (df["abstract_pdf"].isna() | (df["abstract_pdf"].str.strip() == ""))
]
print(f"Weder Keywords noch Abstract: {len(no_abstract_either)}")
# → Diese sind wahrscheinlich schlechte PDFs / Paper ohne Keywords

In [ ]:
suspicious[["title", "ee", "abstract_pdf"]].to_csv("missing.csv")


In [ ]:
df["kw_len"] = df["keywords_pdf"].str.len()

print("Verdächtig kurz (<10 Zeichen):")
print(df[df["kw_len"] < 10][["title", "keywords_pdf"]].to_string())

print("\nVerdächtig lang (>500 Zeichen):")
print(df[df["kw_len"] > 500][["title", "keywords_pdf"]].to_string())

# Und: Enthält der Abstract den ersten Keyword-Begriff? (Sanity Check)
df["sanity"] = df.apply(
    lambda r: str(r["keywords_pdf"]).split(";")[0].strip().lower() 
              in str(r["abstract_pdf"]).lower(), axis=1
)
print(f"\nErster Keyword im Abstract gefunden: {df['sanity'].mean()*100:.1f}%")


In [ ]:
# Zuerst prüfen welches Trennzeichen tatsächlich verwendet wird
sample_kw = df["keywords_pdf"].dropna().head(10)
print(sample_kw.to_list())


In [ ]:
import re

def first_keyword(kw_string):
    parts = re.split(r"[,;|\n]", str(kw_string))
    return parts[0].strip().lower()

df["sanity"] = df.apply(
    lambda r: first_keyword(r["keywords_pdf"]) in str(r["abstract_pdf"]).lower(),
    axis=1
)
print(f"Erster Keyword im Abstract gefunden: {df['sanity'].mean()*100:.1f}%")

Metrik	Wert	Bewertung
Missing (Pipeline-Fehler)	0.8%	✅ excellent
Exact Match (Sample)	85%	✅ gut
ø Similarity (Sample)	0.949	✅ gut
Suspekt kurz/lang	0%	✅
Sanity Check	52%	✅ normal für KR-Domäne